# ASL Detection System — Training & Evaluation Notebook

**VANCO AI Solution Architect Assessment — Use Case 2**

This notebook walks through:
1. Dataset statistics and class distribution
2. Sample image visualisation with annotations
3. Model training (YOLOv8n and YOLOv8s)
4. Evaluation metrics
5. Confusion matrix analysis
6. Inference speed benchmark
7. Model comparison and deployment recommendation

In [ ]:
import os, sys, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

warnings.filterwarnings('ignore')
sys.path.insert(0, 'src')

ASL_CLASSES = ['A','B','C','D','E','F','G','H','I','J','K','L']
print('Environment ready.')

## 1  Dataset Statistics

In [ ]:
# Count images per class per split
split_counts = {}
for split in ['train', 'valid', 'test']:
    split_dir = Path(f'dataset/images/{split}')
    counts = {}
    for cls in ASL_CLASSES:
        imgs = list(split_dir.glob(f'{cls}_*.jpg')) + list(split_dir.glob(f'{cls}_*.png'))
        counts[cls] = len(imgs)
    split_counts[split] = counts

df = pd.DataFrame(split_counts).T
df['Total'] = df.sum(axis=1)
print(df.to_string())

In [ ]:
# Plot class distribution
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors = plt.cm.Set2(np.linspace(0, 1, len(ASL_CLASSES)))
for ax, split in zip(axes, ['train', 'valid', 'test']):
    vals = [split_counts[split].get(c, 0) for c in ASL_CLASSES]
    ax.bar(ASL_CLASSES, vals, color=colors)
    ax.set_title(f'{split.capitalize()} set', fontsize=13)
    ax.set_xlabel('ASL Class')
    ax.set_ylabel('Image count')
    ax.grid(axis='y', alpha=0.3)
plt.suptitle('Class Distribution Across Splits', fontsize=15)
plt.tight_layout()
plt.savefig('outputs/class_distribution.png', dpi=150)
plt.show()

## 2  Sample Images with Annotations

In [ ]:
import cv2
import random

def show_yolo_annotation(img_path, lbl_path, ax, cls_names):
    img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    if lbl_path.exists():
        for line in open(lbl_path).readlines():
            parts = line.strip().split()
            if len(parts) != 5: continue
            cls_id = int(parts[0])
            xc, yc, bw, bh = map(float, parts[1:])
            x1 = int((xc - bw/2) * w)
            y1 = int((yc - bh/2) * h)
            x2 = int((xc + bw/2) * w)
            y2 = int((yc + bh/2) * h)
            cv2.rectangle(img, (x1,y1), (x2,y2), (0,200,100), 2)
            label = cls_names[cls_id] if cls_id < len(cls_names) else str(cls_id)
            cv2.putText(img, label, (x1+2, y1-6), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,200,100), 2)
    ax.imshow(img)
    ax.axis('off')

fig, axes = plt.subplots(2, 6, figsize=(20, 8))
for i, cls in enumerate(ASL_CLASSES):
    imgs = list(Path(f'dataset/images/train').glob(f'{cls}_*.jpg'))[:1]
    ax = axes[i // 6][i % 6]
    if imgs:
        lbl = Path(f'dataset/labels/train') / (imgs[0].stem + '.txt')
        show_yolo_annotation(imgs[0], lbl, ax, ASL_CLASSES)
        ax.set_title(f'Class {cls}', fontsize=11)
    else:
        ax.text(0.5, 0.5, f'{cls}\n(no image)', ha='center', va='center', transform=ax.transAxes)
        ax.axis('off')

plt.suptitle('Sample Annotated Images', fontsize=14)
plt.tight_layout()
plt.savefig('outputs/sample_annotations.png', dpi=150)
plt.show()

## 3  Training

In [ ]:
from train import train_model

# Train YOLOv8n
metrics_n = train_model(
    model_variant='yolov8n',
    epochs=100,
    batch_size=16,
)

In [ ]:
# Plot training curves
from utils import plot_training_curves
results_csv = 'models/asl_yolov8n/results.csv'
if Path(results_csv).exists():
    plot_training_curves(results_csv, 'outputs/training_curves_yolov8n.png')
    from IPython.display import Image
    Image('outputs/training_curves_yolov8n.png')

## 4  Evaluation Metrics

In [ ]:
from ultralytics import YOLO

model = YOLO('models/asl_yolov8n/weights/best.pt')
results = model.val(data='dataset/data.yaml', split='test')

print(f'mAP@0.5:      {results.box.map50:.4f}')
print(f'mAP@0.5:0.95: {results.box.map:.4f}')
print(f'Precision:    {results.box.mp:.4f}')
print(f'Recall:       {results.box.mr:.4f}')

In [ ]:
# Per-class metrics table
if hasattr(results.box, 'ap_class_index'):
    rows = []
    for i, cls_idx in enumerate(results.box.ap_class_index):
        p = float(results.box.p[i])
        r = float(results.box.r[i])
        f1 = 2 * p * r / (p + r + 1e-8)
        rows.append({
            'Class': ASL_CLASSES[cls_idx],
            'Precision': round(p, 4),
            'Recall':    round(r, 4),
            'F1':        round(f1, 4),
            'mAP@0.5':   round(float(results.box.ap50[i]), 4),
        })
    df_cls = pd.DataFrame(rows).set_index('Class')
    display(df_cls.style.background_gradient(subset=['mAP@0.5'], cmap='Greens'))
    df_cls.to_csv('outputs/per_class_metrics.csv')

## 5  Confusion Matrix

In [ ]:
from evaluate import build_confusion_matrix, save_confusion_matrix, error_analysis

cm = build_confusion_matrix(
    model,
    images_dir='dataset/images/test',
    labels_dir='dataset/labels/test',
    conf_thresh=0.5,
)
save_confusion_matrix(cm, output_dir='outputs/evaluation')

errors = error_analysis(cm)
print('\nTop class confusions:')
for conf in errors['top_confusions'][:5]:
    print(f"  {conf['true']} → {conf['predicted']}: {conf['count']} times")

## 6  Inference Benchmark

In [ ]:
from evaluate import benchmark_inference
bench = benchmark_inference(model, img_size=640, n_warmup=20, n_runs=100)

print(f"Device:         {bench['device']}")
print(f"Mean latency:   {bench['latency_mean_ms']} ms")
print(f"p95 latency:    {bench['latency_p95_ms']} ms")
print(f"FPS:            {bench['fps']}")
print(f"Target <50ms:   {'PASS' if bench['meets_target'] else 'FAIL'}")

## 7  Model Comparison

In [ ]:
# Load comparison JSON if available
comp_path = Path('outputs/model_comparison.json')
if comp_path.exists():
    comp = json.load(open(comp_path))
    rows = []
    for variant in ['yolov8n', 'yolov8s']:
        m = comp.get(variant, {})
        rows.append({
            'Model': variant,
            'mAP@0.5': m.get('mAP50', '-'),
            'mAP@0.5:0.95': m.get('mAP50_95', '-'),
            'Precision': m.get('precision', '-'),
            'Recall': m.get('recall', '-'),
            'Size (MB)': m.get('model_size_mb', '-'),
            'Train time (min)': m.get('training_time_min', '-'),
        })
    display(pd.DataFrame(rows).set_index('Model'))
    print(f"\nRecommendation: {comp.get('recommendation', 'N/A')}")
else:
    print('Run python src/train.py --compare to generate comparison.')